# Vietnamese AMR Parser – Colab Gradio Demo
**Model:** MTUP2 · Qwen2.5-7B-Instruct GGUF Q8 · Smatch F1 = 0.646  
**Evaluator:** Groq — Llama 3.3 70B (free)

---
### Trước khi chạy
- Runtime → Change runtime type → T4 GPU → Save.
- Chuẩn bị Hugging Face token có quyền tải model `GiangHuynh16/vn-amr-mtup2-merged`.
- Chuẩn bị Groq API key nếu muốn đánh giá AMR; nếu để trống, demo vẫn parse AMR bình thường.
- Chạy từng cell theo thứ tự. Cell cuối sẽ tạo public Gradio link bằng `share=True`.


In [ ]:
# Cell 1 – Cài packages
!pip install -q llama-cpp-python huggingface_hub groq gradio
print('✓ Packages installed')


In [ ]:
# Cell 2 – Nhập Hugging Face token và download GGUF từ HF Hub
from getpass import getpass
from huggingface_hub import hf_hub_download

HF_REPO_ID = 'GiangHuynh16/vn-amr-mtup2-merged'
HF_FILENAME = 'vn-amr-mtup2-q8.gguf'

hf_token = getpass('Nhập Hugging Face token có quyền tải model: ').strip()
if not hf_token:
    raise ValueError('Cần Hugging Face token để tải model private.')

print('Downloading GGUF (vài phút)...')
MODEL_PATH = hf_hub_download(
    repo_id=HF_REPO_ID,
    filename=HF_FILENAME,
    token=hf_token,
)
print(f'✓ Downloaded: {MODEL_PATH}')


In [ ]:
# Cell 3 – Load model GGUF
from llama_cpp import Llama

print('Loading model onto GPU...')
llm = Llama(
    model_path=MODEL_PATH,
    n_gpu_layers=-1,
    n_ctx=4096,
    verbose=False,
)
print('✓ Model ready!')


In [ ]:
# Cell 4 – Định nghĩa hàm parse, postprocess và đánh giá
import re
from collections import Counter
from groq import Groq

SYSTEM_PROMPT = """Bạn là chuyên gia phân tích AMR (Abstract Meaning Representation) cho tiếng Việt.

QUY TẮC QUAN TRỌNG:
1. Chỉ dùng từ tiếng Việt trong câu gốc làm concept. KHÔNG tự tạo từ mới.
2. Concept AMR chuẩn được phép dùng (tiếng Anh): person, name, now, after, before, and, or, contrast-01, obligate-01, possible-01, rate-entity-91, temporal-quantity, percentage-entity, multi-sentence, have-rel-role-91, have-org-role-91, amr-unknown, province, city, mountain, country, company, ethnic-group, date-entity, date-interval, monetary-quantity, about
3. :classifier cho lượng từ: người, anh, chị, cái, con, chiếc, đứa, ngôi, căn, cùi, ngọn, chú, cánh
4. :polarity - cho phủ định, :modality + cho được, :modality - cho bị
5. :wiki - cho tên riêng người/địa danh
6. :pivot cho trạng thái/tính từ, :agent cho hành động
7. "hiện nay", "đến nay", "nay" → (n / now)
8. Từ ghép giữ nguyên: "làm việc", "chịu thua", "thoát nghèo", "vận động"
9. Đảm bảo ngoặc cân bằng, mỗi biến DUY NHẤT trong AMR
10. :prep cho giới từ: về, từ, ở, vào, trong, cho, tại, bởi, vì, để, với
11. :domain cho tính từ/trạng thái khi subject không chủ động hành động
12. "X nói/khẳng định/cho biết: ..." → (s / say-01 :agent X :topic(mệnh đề))"""

GROQ_EVAL_PROMPT = """Bạn là chuyên gia đánh giá AMR (Abstract Meaning Representation) tiếng Việt. Hãy chấm nghiêm khắc, không khen chung chung và không cho điểm cao nếu AMR chỉ đúng bề mặt.

Câu gốc: {sentence}

AMR được sinh ra:
{amr}

Nguyên tắc chấm:
- Chỉ đánh giá AMR được cung cấp, không tự sửa AMR trước khi chấm.
- So sánh sát với nghĩa câu gốc tiếng Việt: vị ngữ chính, vai nghĩa, phủ định, thời gian, địa điểm, tên riêng, số lượng, quan hệ nguyên nhân/điều kiện/tương phản.
- Relation sai vai (:agent, :theme, :pivot, :domain, :prep, :mod, :name, :wiki...) phải trừ điểm rõ ràng, kể cả khi concept nhìn có vẻ đúng.
- AMR cân bằng ngoặc nhưng sai ngữ nghĩa vẫn không được điểm cao.
- Nếu thiếu ý quan trọng, thêm concept không có trong câu, dùng biến trùng, hoặc chọn root sai thì phải nêu lỗi cụ thể.

Trả lời ngắn gọn theo đúng format sau:

1. Cú pháp: Đúng/Sai – nêu lỗi nếu có, gồm ngoặc, biến, cấu trúc relation.
2. Ngữ nghĩa: Tốt/Chấp nhận được/Có lỗi – đánh giá root concept và các vai nghĩa chính; nêu relation sai nếu có.
3. Độ phủ: Đầy đủ/Thiếu/Thừa – liệt kê từ hoặc ý bị bỏ sót/thêm sai nếu có.
4. Kết luận: 1–2 câu tổng thể.
5. Điểm: X/10

Hướng dẫn điểm:
- 9–10: Gần như đúng hoàn toàn cả cú pháp, root, vai nghĩa và độ phủ.
- 7–8: Đúng phần lớn nhưng có lỗi nhỏ về relation, format hoặc thiếu ý phụ.
- 5–6: Hiểu được ý chính nhưng có lỗi đáng kể về root, vai nghĩa hoặc thiếu ý quan trọng.
- 3–4: Nhiều lỗi ngữ nghĩa/cấu trúc, chỉ giữ được vài concept đúng.
- 1–2: Sai nghiêm trọng, không phản ánh câu gốc hoặc AMR hầu như không hợp lệ."""


def extract_balanced_amr(text, start_pos):
    paren_start = text.find('(', start_pos)
    if paren_start == -1:
        return None

    depth = 0
    for i in range(paren_start, len(text)):
        if text[i] == '(':
            depth += 1
        elif text[i] == ')':
            depth -= 1
            if depth == 0:
                return text[paren_start:i + 1]

    last = text.rfind(')', paren_start)
    if last != -1:
        fragment = text[paren_start:last + 1]
        opens, closes = fragment.count('('), fragment.count(')')
        if opens > closes:
            fragment += ')' * (opens - closes)
        return fragment
    return None


def is_probable_amr(fragment):
    if not fragment:
        return False
    return re.match(r'^\s*\(\s*[A-Za-zÀ-ỹ_][\wÀ-ỹ_-]*\s*/\s*[^\s()]+', fragment) is not None


def extract_amr_from_response(response):
    markers = [
        '[STEP 4: AMR_WITH_VAR]',
        '[STEP 3: AMR_WITH_VAR]',
        '[STEP 2: AMR_WITH_VAR]',
        '[STEP 1: AMR_WITH_VAR]',
        '[STEP 4: AMR_NO_VAR]',
        '[STEP 3: AMR_NO_VAR]',
        '[STEP 2: AMR_NO_VAR]',
        '[STEP 1: AMR_NO_VAR]',
        'AMR_WITH_VAR:',
        'AMR_NO_VAR:',
        'AMR:',
    ]

    for marker in markers:
        pos = response.rfind(marker)
        if pos == -1:
            continue
        amr = extract_balanced_amr(response, pos)
        if is_probable_amr(amr):
            return amr

    # Fallback cho GGUF/llama.cpp khi model trả trực tiếp AMR mà không có marker.
    search_pos = 0
    while True:
        paren_pos = response.find('(', search_pos)
        if paren_pos == -1:
            return None
        amr = extract_balanced_amr(response, paren_pos)
        if is_probable_amr(amr):
            return amr
        search_pos = paren_pos + 1


def postprocess_amr(amr):
    amr = amr.strip()
    if not amr:
        return '(a / amr-unknown)'

    for var, count in Counter(re.findall(r'\((\w+)\s*/', amr)).items():
        if count > 1:
            index = [0]

            def replace_definition(match, v=var):
                index[0] += 1
                return f'({v}{index[0]} /'

            amr = re.sub(r'\(' + re.escape(var) + r'\s*/', replace_definition, amr)
            amr = re.sub(r'(?<!\w)' + re.escape(var) + r'(?!\s*/)(?!\w)', f'{var}1', amr)

    amr = re.sub(r':polarity\s*\(\s*-\s*\)', ':polarity -', amr)
    amr = re.sub(r':wiki\s*\(\s*-\s*\)', ':wiki -', amr)

    opens, closes = amr.count('('), amr.count(')')
    if opens > closes:
        amr += ')' * (opens - closes)
    elif closes > opens:
        for _ in range(closes - opens):
            last = amr.rfind(')')
            if last != -1:
                amr = amr[:last] + amr[last + 1:]

    return re.sub(r'\s+', ' ', amr).strip()


def format_amr_pretty(amr_flat):
    result = []
    indent = 0
    i = 0
    while i < len(amr_flat):
        char = amr_flat[i]
        if char == '(':
            result.append('\n' + '    ' * indent + '(')
            indent += 1
        elif char == ')':
            indent = max(0, indent - 1)
            result.append(')')
        elif char == ':' and i > 0 and amr_flat[i - 1] == ' ':
            result.append('\n' + '    ' * indent + ':')
            i += 1
            continue
        else:
            result.append(char)
        i += 1
    return ''.join(result).strip()


def parse_sentence(sentence):
    prompt = (
        f'<|im_start|>system\n{SYSTEM_PROMPT}<|im_end|>\n'
        f'<|im_start|>user\n{sentence.strip()}<|im_end|>\n'
        f'<|im_start|>assistant\n'
    )
    output = llm(
        prompt,
        max_tokens=1536,
        temperature=0,
        repeat_penalty=1.1,
        stop=['<|im_end|>'],
    )
    response = output['choices'][0]['text']

    amr = extract_amr_from_response(response)
    if not amr:
        debug_response = response if response.strip() else '[EMPTY_RESPONSE] Model không sinh output sau prompt.'
        return '(a / amr-unknown)', debug_response

    amr = postprocess_amr(amr)
    return re.sub(r'\s+', ' ', amr).strip(), response


def evaluate_with_groq(sentence, amr, api_key):
    if not api_key.strip():
        return '⚠️ Chưa nhập Groq API key. AMR đã được parse, nhưng chưa đánh giá tự động.'
    if amr == '(a / amr-unknown)':
        return '⚠️ AMR không hợp lệ, không thể đánh giá.'

    try:
        client = Groq(api_key=api_key.strip())
        prompt = GROQ_EVAL_PROMPT.format(sentence=sentence, amr=amr)
        response = client.chat.completions.create(
            model='llama-3.3-70b-versatile',
            messages=[{'role': 'user', 'content': prompt}],
            max_tokens=1024,
        )
        return response.choices[0].message.content
    except Exception as exc:
        return f'❌ Lỗi Groq API: {exc}'


print('✓ Tất cả hàm sẵn sàng.')


In [ ]:
# Cell 5 – Chạy Gradio UI
import gradio as gr

EXAMPLES = [
    'Tôi yêu Hà Nội.',
    'Anh ấy không đi làm hôm nay.',
    'Cô giáo dạy học sinh viết văn.',
    'Hiện nay xã có 68 tổ nhân dân.',
    'Chủ trương tốt nhưng dân không hiểu.',
]


def run(sentence, api_key):
    if not sentence.strip():
        return '', '', '', '⚠️ Vui lòng nhập câu.'

    amr_flat, raw = parse_sentence(sentence)
    amr_pretty = format_amr_pretty(amr_flat)
    evaluation = evaluate_with_groq(sentence, amr_flat, api_key)
    return amr_flat, amr_pretty, raw, evaluation


with gr.Blocks(title='Vietnamese AMR Parser', theme=gr.themes.Soft()) as demo:
    gr.Markdown("""
# Vietnamese AMR Parser
**Model:** MTUP2 — Qwen2.5-7B-Instruct GGUF Q8 &nbsp;|&nbsp; Smatch F1 = 0.646  
**Evaluator:** Groq — Llama 3.3 70B (free)
""")
    with gr.Row():
        with gr.Column(scale=3):
            sentence_in = gr.Textbox(
                label='Câu tiếng Việt',
                placeholder='Nhập câu cần parse...',
                lines=2,
            )
            api_key_in = gr.Textbox(
                label='Groq API Key (free tại console.groq.com)',
                placeholder='gsk_...',
                type='password',
            )
            btn = gr.Button('Parse & Evaluate', variant='primary')
        with gr.Column(scale=1):
            gr.Markdown('**Ví dụ nhanh:**')
            for ex in EXAMPLES:
                gr.Button(ex, size='sm').click(fn=lambda s=ex: s, outputs=sentence_in)

    with gr.Row():
        amr_flat_out = gr.Textbox(label='AMR (single line)', lines=3)
        amr_pretty_out = gr.Textbox(label='AMR (indented)', lines=10)

    eval_out = gr.Markdown()

    with gr.Accordion('Raw model output (debug)', open=False):
        raw_out = gr.Textbox(label='Full response', lines=15)

    btn.click(
        fn=run,
        inputs=[sentence_in, api_key_in],
        outputs=[amr_flat_out, amr_pretty_out, raw_out, eval_out],
    )
    sentence_in.submit(
        fn=run,
        inputs=[sentence_in, api_key_in],
        outputs=[amr_flat_out, amr_pretty_out, raw_out, eval_out],
    )

# share=True tạo public Gradio link để test ngoài Colab.
demo.queue().launch(share=True, debug=False)
